In [0]:
%python
dbutils.widgets.removeAll()

In [0]:
create widget text storageName default "adlsproject1";

In [0]:
%python
storageName = dbutils.widgets.get("storageName")

In [0]:
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-raw`
URL 'abfss://raw@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas raw del Data Lake';

In [0]:
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-bronze`
URL 'abfss://bronze@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas bronze del Data Lake';

In [0]:
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-silver`
URL 'abfss://silver@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas silver del Data Lake';

In [0]:
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-golden`
URL 'abfss://golden@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas golden del Data Lake';

In [0]:
DROP CATALOG IF EXISTS catalog_au CASCADE;

In [0]:
CREATE CATALOG IF NOT EXISTS catalog_au
MANAGED LOCATION 'abfss://metastore@${storageName}.dfs.core.windows.net/'
COMMENT 'Catalogo para la arquitectura medallion del ambiente de dev';

In [0]:
DROP SCHEMA IF EXISTS catalog_au.raw;
DROP SCHEMA IF EXISTS catalog_au.bronze;
DROP SCHEMA IF EXISTS catalog_au.silver;
DROP SCHEMA IF EXISTS catalog_au.golden;

In [0]:
%python
dbutils.fs.rm(f"abfss://bronze@{storageName}.dfs.core.windows.net/",True)
dbutils.fs.rm(f"abfss://silver@{storageName}.dfs.core.windows.net/",True)
dbutils.fs.rm(f"abfss://golden@{storageName}.dfs.core.windows.net/",True)

In [0]:
CREATE SCHEMA IF NOT EXISTS catalog_au.raw;
CREATE SCHEMA IF NOT EXISTS catalog_au.bronze;
CREATE SCHEMA IF NOT EXISTS catalog_au.silver;
CREATE SCHEMA IF NOT EXISTS catalog_au.golden;

###Tablas Bronze

In [0]:
CREATE TABLE IF NOT EXISTS catalog_au.bronze.order_products__train (
  order_id integer,
  product_id integer,
  add_to_cart_order integer,
  reordered integer,
  ingestion_date timestamp
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/order_products__train"

In [0]:
CREATE TABLE IF NOT EXISTS catalog_au.bronze.products (
  product_id integer,
  product_name string,
  aisle_id integer,
  department_id integer,
  ingestion_date timestamp
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/products"

###Tablas Silver

In [0]:
CREATE TABLE IF NOT EXISTS catalog_au.silver.products_orders_enriched (
  order_id integer,
  product_id integer,
  product_name string,
  aisle_id integer,
  department_id integer,
  add_to_cart_order integer,
  cart_position_category string,
  reordered integer,
  reorder_desc string,
  is_organic string,
  ingestion_date timestamp
)
USING DELTA
LOCATION "abfss://silver@${storageName}.dfs.core.windows.net/products_orders_enriched"

###Tablas Golden

In [0]:
CREATE TABLE IF NOT EXISTS catalog_au.golden.kpis_sales_summary_partitioned (
  department_id integer,
  is_organic string,
  total_orders long,
  total_reorders long,
  avg_add_to_cart_order double
)
USING DELTA
LOCATION "abfss://golden@${storageName}.dfs.core.windows.net/kpis_sales_summary_partitioned"
PARTITIONED BY (is_organic)